# Prefect

This is a tutorial to use the Prefect cluster from Jupyter, without Dask.

See the associated Python module: [my_prefect.py](./my_prefect.py)

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *

In [ ]:
%%bash
prefect block ls
echo -e "\nNOTE: you can see the block details and credentials by running e.g.: prefect block inspect s3-bucket/s3"

In [ ]:
# Other imports
import json
import os
import prefect
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import resources

# Set environment variables for the client
os.environ["HELLO_FROM"] = "client"

# NOTE: when deploying, the prefect flows and tasks must be implemented in a separate python module.
# We cannot implement them directly in jupyter cells.
import my_prefect

# Data to test the example flow
my_data = [
    "PrefectHQ/prefect",
    "pydantic/pydantic",
    "huggingface/transformers"
] * 5 # repeat the data

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

## 1. Implement the `quickstart` tutorial
See: https://docs.prefect.io/v3/get-started/quickstart

When calling the flow as a normal python function, the flow and tasks are run by Prefect on your local client environment = your Jupyter or terminal.

This is the easiest way to test your Prefect code because the same environment, Python interpreter and files are shared between your client, flow and tasks. But this is less performant because your tasks are not distributed on the cluster.

In [ ]:
# Import the module, or reload it if you changed its source code
import my_prefect
reload(my_prefect)

# Run the flow
my_prefect.flow_show_stars(my_data)

<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs above that when we call the flow from Python code, your module main code, flow and tasks are run by the client. No Prefect workers are involved.
  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.

## 2. Run flows in local processes
See: https://docs.prefect.io/v3/deploy/run-flows-in-local-processes

Create a deployment for a flow by calling the `serve` method.

As for the quickstart above, the same environment, Python interpreter and files are shared between your client, flow and tasks.

In [ ]:
# Deploy the flow
task = prefect_utils.hack_for_jupyter( # we need a hack to deploy from jupyter
    my_prefect.flow_show_stars.serve,
    name="tuto-serve-python",
    tags=["tutorial"],
)
serve_python = "flow-show-stars/tuto-serve-python"
await prefect_utils.wait_for_deployment(serve_python)

In [ ]:
%%bash -s "$serve_python" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2" --watch

In [ ]:
from prefect.settings import PREFECT_UI_URL
print(f"""
########
# NOTE #
########

Don't use the internal domain from the logs above: {PREFECT_UI_URL.value()!r}, use the public domain instead: {os.environ['RSPY_PREFECT_URL']}
""")

<div class="alert alert-info" role="alert">
Notes:

  1. The deployment is made in your client (=your Jupyter or terminal). If you kill your client, the deployment will not be available anymore.
  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.
  1. Check in the logs that when we run the flow with `.serve`, your module main code, flow and tasks are run by the client. No Prefect workers are involved.

## 3. Deploy flows with Python

See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/deploy-via-python

Prefect offers a flexible way to deploy flows to dynamic infrastructure using the Python SDK. This approach allows you to target specific work pools and utilize dynamically provisioned infrastructure.

This is easier to deploy than with YAML (see next section) but less complete (e.g. cannot run additional scripts or pip install ...)

### Deploy the source code

You want to deploy flows and tasks from you local source code... but this source code doesn't exist in the prefect worker container. So you need to store it somewhere and transfer it. 

We can use this project git repository but this is not very flexible (as for now you cannot even specify a git branch when deploying from python).

Another solution is to transfer the source code via the S3 bucket using prefect blocks.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

# Use a specific secret block on the bucket for this subfolder
code_bucket, os.environ["SHARE_BUCKET"] = await get_share_bucket(s3_code_folder)

if local_mode:
    print (f"S3 SeaweedFS dashboard: http://localhost:9101 with user=seaweedfs password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{code_bucket.bucket_name}/{code_bucket.bucket_folder}'")

# Upload local directory and resources contents
await code_bucket.put_directory(local_path = ".", to_path = ".")
await code_bucket.put_directory(local_path = resources.__path__[0], to_path = "resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = code_bucket.bucket_folder

In [ ]:
# Deploy the flow
flow = await prefect.flow.from_source(
    source=code_bucket,
    entrypoint="my_prefect.py:flow_show_stars",
)
await flow.deploy(
    name="tuto-deploy-python",
    # We can use any work pool
    work_pool_name=os.environ["PREFECT_WORK_POOL_SANDBOX"], 
    tags=["tutorial"],
    ignore_warnings=True,
    job_variables={"env": {"HELLO_FROM": "prefect"}}, # pass env vars
)
deploy_python = "flow-show-stars/tuto-deploy-python"
await prefect_utils.wait_for_deployment(deploy_python)

In [ ]:
%%bash -s "$deploy_python" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --param github_repos="$2" --watch

<div class="alert alert-info" role="alert">
Notes:

  1. The deployment is made in the prefect server container. If you kill your client, the deployment will still be available.
  1. In the Prefect dashboard (see link above), find your deployment and your run, check its graph and logs.
  1. You can also trigger a run from the dashboard deployment page.
  1. Check in the logs that when we deploy the flow:
      1. Your module main code (outside functions) is run by both the client and Prefect workers.
      1. Your module flow and tasks are run only by the Prefect workers.
  1. If you want to check the IP adresses:
      1. On kubernetes, you can run the `kubectl describe` command to check a pod IP address.
      1. In local mode, use: `docker inspect <container_id> | grep IPAddress`
  1. All these notes are also true when you deploy with YAML (see next section).

## 4. Define deployments with YAML
See: https://docs.prefect.io/v3/deploy/infrastructure-concepts/prefect-yaml

Use YAML to schedule and trigger flow runs and manage your code and deployments.

This is the most complete way to deploy your flow and tasks.

### 4.1 Deploy with YAML from git repository

See the full yaml file: [deploy-yaml-git.yaml](./deploy-yaml-git.yaml)

As above, we need to store and transfer our flow and tasks from our local source code, saved in the project git repository. We can tell prefect to pull the source code from there before deploying it. 

This is the easiest way to transfer your code, but you must be careful to deploy your right git branch name, and push your code to git after each local modification.
```yaml
pull:
- prefect.deployments.steps.git_clone:
    repository: https://github.com/org/repo.git
    branch: main
    credentials: "{{ prefect.blocks.github-credentials.my-credentials }}"
```

We can pass environment variables with:
```yaml
deployments:
- name: ...
  work_pool:
    job_variables:
        env:
         HELLO_FROM: "prefect"
```

We can install additional modules before running the flow either with:
```yaml
pull:
- prefect.deployments.steps.git_clone:
    id: clone-step # needed to be referenced in subsequent steps
    repository: https://github.com/org/repo.git
- prefect.deployments.steps.pip_install_requirements:
    directory: "{{ clone-step.directory }}" # `clone-step` is a user-provided `id` field
    requirements_file: requirements.txt
```
or:
```yaml
- prefect.deployments.steps.run_shell_script:
    script: pip install # ...
```
In this example, we will install the module `argh`.

NOTES:

  1. These additional installations will persist in the worker container, which means that they will still be there for other runs or deployments.
  1. We need to trigger the deployment from the git root folder and use the relative path to the yaml file. The entrypoint from the yaml file must also use a relative file from the git root folder:

```yaml
deployments:
- entrypoint: ./notebooks/tutorials/prefect+dask/my_prefect.py:flow_show_stars
```

In [ ]:
%%bash
# Go to the git root folder
cd ../../..
# Deploy the flow
prefect --no-prompt deploy --prefect-file "notebooks/tutorials/prefect+dask/deploy-yaml-git.yaml"

In [ ]:
deploy_yaml_git = "flow-show-stars/tuto-deploy-yaml-git"
await prefect_utils.wait_for_deployment(deploy_yaml_git)

In [ ]:
%%bash -s "$deploy_yaml_git" "$my_data_str"
# Trigger a run for this flow from the command line. Test that the 'argh' module was installed.
prefect deployment run "$1" --param github_repos="$2" --param test_pip="argh" --watch

### 4.2 Deploy with YAML from S3 bucket

See the full yaml file: [deploy-yaml-s3.yaml](./deploy-yaml-s3.yaml)

As when deploying flows with Python, we can transfer our local source code via the S3 bucket using prefect blocks. We can also transfer additional wheel files to be installed before running the flow. In this example, we will install the module `emoji`:

```yaml
pull:
- prefect.deployments.steps.run_shell_script:
    script: |
        python -c "from prefect.filesystems import RemoteFileSystem; RemoteFileSystem.load('s3').get_directory('users/jovyan', '.')"        
        pip install emoji --find-links ./wheels
```

NOTE: here also, these additional installations will persist in the worker container, which means that they will still be there for other runs or deployments.


In [ ]:
# Upload local directory contents as above, in case they have changed
await code_bucket.put_directory(local_path = ".", to_path = ".")
await code_bucket.put_directory(local_path = resources.__path__[0], to_path = "resources")

# Do the same with a wheel file. First download it.
whl_dir = "/tmp/emoji"
!rm -rf $whl_dir && mkdir -p $whl_dir && pip download --dest $whl_dir emoji

# Then upload its directory to S3
await code_bucket.put_directory(local_path = whl_dir, to_path = "wheels")

# Pass the full S3 code folder
os.environ["S3_CODE_FOLDER"] = code_bucket.bucket_folder

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./deploy-yaml-s3.yaml"

In [ ]:
deploy_yaml_s3 = "flow-show-stars/tuto-deploy-yaml-s3"
await prefect_utils.wait_for_deployment(deploy_yaml_s3)

In [ ]:
%%bash -s "$deploy_yaml_s3" "$my_data_str"
# Trigger a run for this flow from the command line. Test that the 'emoji' module was installed.
prefect deployment run "$1" --param github_repos="$2" --param test_pip="emoji" --watch